In [0]:
SET datasets.path='dbfs:/mnt/demo-datasets/bookstore';

In [0]:
SET datasets.path=dbfs:/mnt/demo-datasets/bookstore;
CREATE OR REFRESH STREAMING LIVE TABLE orders_raw_live
COMMENT "The raw books orders, ingested from orders-raw"
AS SELECT * FROM cloud_files("dbfs:/mnt/demo-datasets/bookstore/orders-json-raw", "json",
                             map("cloudFiles.inferColumnTypes", "true"))

In [0]:
SET datasets.path=dbfs:/mnt/demo-datasets/bookstore;
CREATE OR REFRESH LIVE TABLE customers_live
COMMENT "The customers lookup table, ingested from customers-json"
AS SELECT * FROM json.`dbfs:/mnt/demo-datasets/bookstore/customers-json`

In [0]:
CREATE OR REFRESH STREAMING LIVE TABLE orders_cleaned (
  CONSTRAINT valid_order_number EXPECT (order_id IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT "The cleaned books orders with valid order_id"
AS
  SELECT order_id, quantity, o.customer_id, c.profile:first_name as f_name, c.profile:last_name as l_name,
         cast(from_unixtime(order_timestamp, 'yyyy-MM-dd HH:mm:ss') AS timestamp) order_timestamp, o.books,
         c.profile:address:country as country
  FROM STREAM(LIVE.orders_raw_live) o
  LEFT JOIN LIVE.customers_live c
    ON o.customer_id = c.customer_id

In [0]:
CREATE OR REFRESH LIVE TABLE cn_daily_customer_books
COMMENT "Daily number of books per customer in China"
AS
  SELECT customer_id, f_name, l_name, date_trunc("DD", order_timestamp) order_date, sum(quantity) books_counts
  FROM LIVE.orders_cleaned
  WHERE country = "China"
  GROUP BY customer_id, f_name, l_name, date_trunc("DD", order_timestamp)

In [0]:
CREATE OR REFRESH LIVE TABLE fr_daily_customer_books
COMMENT "Daily number of books per customer in France"
AS
  SELECT customer_id, f_name, l_name, date_trunc("DD", order_timestamp) order_date, sum(quantity) books_counts
  FROM LIVE.orders_cleaned
  WHERE country = "France"
  GROUP BY customer_id, f_name, l_name, date_trunc("DD", order_timestamp)